# Prerequisite 06 · PyTorch：从 tensor 到完整训练循环

[上一章：神经网络从零开始](Prerequisite05B_Neural_Network_From_Scratch.ipynb) · [下一章：卷积网络与序列模型](Prerequisite06B_CNN_and_Sequences.ipynb)

**目标：**理解 tensor、autograd、`nn.Module`、optimizer、`detach/no_grad`、device 和概率分布；独立写出一个小型训练循环。


## 1. Tensor 与 NumPy 的联系

Tensor 同样有 shape、dtype 和广播，还能记录计算图并在 CPU/GPU 上计算。RL 中通常把 NumPy observation 转成 float tensor，再送入网络。


In [1]:
import torch

torch.manual_seed(6)
states = torch.tensor([[.1, .2, -.3], [0., .4, .5]], dtype=torch.float32)
linear = torch.nn.Linear(3, 2)
q_values = linear(states)
actions = torch.tensor([1, 0])
chosen_q = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
print("Q shape:", tuple(q_values.shape), "chosen Q shape:", tuple(chosen_q.shape))


Q shape: (2, 2) chosen Q shape: (2,)


## 2. Autograd：计算图上的链式法则

`loss.backward()` 把梯度累积到参数的 `.grad`；`optimizer.step()` 才真正修改参数。默认梯度会累加，所以每轮先 `zero_grad()`。


In [2]:
w = torch.tensor(1.5, requires_grad=True)
x, y = torch.tensor(3.0), torch.tensor(7.0)
loss = (w*x - y)**2
loss.backward()
print("loss:", loss.item(), "dL/dw:", w.grad.item())
assert abs(w.grad.item() - 2*(1.5*3-7)*3) < 1e-6


loss: 6.25 dL/dw: -15.0


## 3. 完整训练循环

顺序固定为：准备 batch → forward → loss → `zero_grad` → `backward` → `step`。评估时使用 `model.eval()` 与 `torch.no_grad()`。


In [3]:
x = torch.linspace(-1, 1, 100).unsqueeze(1)
y = 2.5*x - .4 + .08*torch.randn_like(x)
model = torch.nn.Sequential(torch.nn.Linear(1, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=.03)
history = []
for _ in range(120):
    prediction = model(x)
    loss = torch.nn.functional.mse_loss(prediction, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append(loss.item())

model.eval()
with torch.no_grad():
    final_mse = torch.nn.functional.mse_loss(model(x), y).item()
print("first/final loss:", round(history[0], 4), round(final_mse, 4))
assert final_mse < history[0] * .1


first/final loss: 2.8891 0.0088


## 4. detach、no_grad 与概率分布

- `detach()` 返回共享数据但不继续追踪当前计算图，常用于 TD target 或 advantage target。
- `no_grad()` 在整个代码块中关闭梯度，适合 target network 和评估。
- `torch.distributions.Categorical/Normal` 提供 `sample`、`log_prob`、`entropy`，是策略梯度的基础。


In [4]:
logits = torch.tensor([.2, 1.0, -.4], requires_grad=True)
distribution = torch.distributions.Categorical(logits=logits)
action = distribution.sample()
objective = -distribution.log_prob(action) * torch.tensor(1.5)
objective.backward()
print("sample/log_prob/entropy:", action.item(), distribution.log_prob(action).item(), distribution.entropy().item())
print("logit gradient:", logits.grad)


sample/log_prob/entropy: 1 -0.528228759765625 0.9437533020973206
logit gradient: tensor([ 0.3974, -0.6155,  0.2181])


## 5. 用 PyTorch 写一个分类 MLP（含 mini-batch）

真实训练很少一次用全部数据，而是每次取一个 **mini-batch**。`TensorDataset + DataLoader` 负责打乱和分批，
DQN 从 replay buffer 抽 batch 的思路与此相同。下面在一个“两个月牙”形状的二分类数据上训练 MLP。


In [5]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

def make_moons(n, noise=0.15):
    t = torch.rand(n) * math.pi
    upper = torch.stack([torch.cos(t), torch.sin(t)], dim=1)
    lower = torch.stack([1 - torch.cos(t), 0.5 - torch.sin(t)], dim=1)
    x = torch.cat([upper, lower]) + noise * torch.randn(2 * n, 2)
    y = torch.cat([torch.zeros(n), torch.ones(n)]).long()
    return x, y

x_train, y_train = make_moons(400)
x_val, y_val = make_moons(200)
loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)

model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()       # 输入 logits (B,C) 和整数标签 (B,)

for epoch in range(40):
    model.train()
    for xb, yb in loader:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    if epoch % 10 == 9:
        model.eval()
        with torch.no_grad():
            acc = (model(x_val).argmax(dim=1) == y_val).float().mean().item()
        print(f"epoch {epoch+1:>2}: 最后一个 batch 的 loss {loss.item():.3f}, 验证准确率 {acc:.3f}")
assert acc > 0.9


epoch 10: 最后一个 batch 的 loss 0.023, 验证准确率 0.990


epoch 20: 最后一个 batch 的 loss 0.001, 验证准确率 0.993


epoch 30: 最后一个 batch 的 loss 0.020, 验证准确率 0.990


epoch 40: 最后一个 batch 的 loss 0.016, 验证准确率 0.993


## 6. 保存与加载、target network 的“复制”

- 保存模型通常只保存参数：`torch.save(model.state_dict(), path)`；加载时先建同结构模型，再 `load_state_dict`。
- DQN 的 target network 就是“定期把在线网络的参数复制过去”：`target.load_state_dict(online.state_dict())`。
- DDPG/SAC 用**软更新**：$\theta_{target}\leftarrow\tau\theta+(1-\tau)\theta_{target}$，每步只靠近一点点（本课练习 4）。


In [6]:
import copy, os, tempfile

online = nn.Linear(4, 2)
target = copy.deepcopy(online)                 # 结构和参数都复制一份
with torch.no_grad():
    online.weight += 1.0                       # 模拟在线网络被训练改动
print("复制前差异:", (online.weight - target.weight).abs().max().item())
target.load_state_dict(online.state_dict())    # 硬更新（DQN 每 N 步一次）
print("复制后差异:", (online.weight - target.weight).abs().max().item())

path = os.path.join(tempfile.gettempdir(), "tiny_model.pt")
torch.save(online.state_dict(), path)
restored = nn.Linear(4, 2); restored.load_state_dict(torch.load(path, weights_only=True))
assert torch.equal(restored.weight, online.weight)
print("保存/加载成功:", path)


复制前差异: 1.0
复制后差异: 0.0
保存/加载成功: /tmp/tiny_model.pt


## 自动批改练习

每道题先读题、再写代码、最后运行同一格。看到 `[通过]` 才算完成；`[未通过]` 会说明哪里不对。
参考答案默认折叠，建议独立尝试至少 10 分钟后再看。


In [7]:
# 自动检查工具：每道练习末尾都会调用 check(test_xxx)。
# 未完成或有错时只打印提示，不会中断 Run All。
def check(test):
    try:
        test()
    except NotImplementedError:
        print(f"[未完成] {test.__name__}：把 raise NotImplementedError 换成你的代码，再运行本格。")
    except AssertionError as error:
        print(f"[未通过] {test.__name__}：{error}")
    except Exception as error:  # 代码本身报错，也给出可读提示
        print(f"[代码报错] {test.__name__}：{type(error).__name__}: {error}")
    else:
        print(f"[通过] {test.__name__}")


### 练习 1：用 gather 取已选动作的 Q 值

实现 `chosen_q(q_values, actions)`：`q_values` 为 `(B,A)` 张量，`actions` 为 `(B,)` 的 `long` 张量，返回 `(B,)`。要求使用 `gather`，并保持梯度可以传回 `q_values`。

<details><summary>提示（卡住 10 分钟再看）</summary>

`q_values.gather(1, actions.unsqueeze(1)).squeeze(1)`。

</details>


In [8]:
import torch

def chosen_q(q_values, actions):
    raise NotImplementedError


# ---- 自动检查（不用修改） ----
import torch

def test_chosen_q():
    q = torch.tensor([[1., 4., 2.], [5., 0., 3.]], requires_grad=True)
    out = chosen_q(q, torch.tensor([1, 2]))
    assert tuple(out.shape) == (2,), f"应为 (2,)，得到 {tuple(out.shape)}"
    assert torch.allclose(out, torch.tensor([4., 3.])), f"应为 [4, 3]，得到 {out}"
    out.sum().backward()
    assert torch.equal(q.grad, torch.tensor([[0., 1., 0.], [0., 0., 1.]])), "梯度只应流向被选中的位置"

check(test_chosen_q)


[未完成] test_chosen_q：把 raise NotImplementedError 换成你的代码，再运行本格。


<details><summary>参考答案（先独立完成，再对照）</summary>

```python
import torch

def chosen_q(q_values, actions):
    return q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
```

</details>


### 练习 2：构建 MLP

实现 `make_mlp(sizes)`：例如 `sizes=[4, 64, 64, 2]` 返回 `Linear(4,64)→ReLU→Linear(64,64)→ReLU→Linear(64,2)`，**最后一层后面不加激活**（Q 值可以是负数）。

<details><summary>提示（卡住 10 分钟再看）</summary>

循环 `zip(sizes[:-1], sizes[1:])` 收集层，除最后一层外都追加 `nn.ReLU()`，最后 `nn.Sequential(*layers)`。

</details>


In [9]:
import torch.nn as nn

def make_mlp(sizes):
    raise NotImplementedError


# ---- 自动检查（不用修改） ----
import torch
import torch.nn as nn

def test_make_mlp():
    net = make_mlp([4, 64, 64, 2])
    kinds = [type(m).__name__ for m in net]
    assert kinds == ["Linear", "ReLU", "Linear", "ReLU", "Linear"], f"层结构不对：{kinds}"
    assert tuple(net(torch.zeros(5, 4)).shape) == (5, 2), "输入 (5,4) 应输出 (5,2)"

check(test_make_mlp)


[未完成] test_make_mlp：把 raise NotImplementedError 换成你的代码，再运行本格。


<details><summary>参考答案（先独立完成，再对照）</summary>

```python
import torch.nn as nn

def make_mlp(sizes):
    layers = []
    for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(n_in, n_out))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)
```

</details>


### 练习 3：完整训练循环

实现 `train_regressor(model, x, y, steps, lr)`：用 Adam 和 MSE 训练 `model`，返回**每一步的 loss 列表**（Python float）。
必须包含完整五步：forward → loss → `zero_grad` → `backward` → `step`。


In [10]:
import torch

def train_regressor(model, x, y, steps, lr):
    raise NotImplementedError


# ---- 自动检查（不用修改） ----
import torch

def test_train_regressor():
    torch.manual_seed(0)
    x = torch.linspace(-1, 1, 64).unsqueeze(1); y = torch.sin(3 * x)
    model = torch.nn.Sequential(torch.nn.Linear(1, 32), torch.nn.Tanh(), torch.nn.Linear(32, 1))
    history = train_regressor(model, x, y, steps=300, lr=0.01)
    assert len(history) == 300 and isinstance(history[0], float), "应返回 300 个 float"
    assert history[-1] < history[0] * 0.1, f"loss 应下降到初始的 10% 以下：{history[0]:.3f} → {history[-1]:.3f}"

check(test_train_regressor)


[未完成] test_train_regressor：把 raise NotImplementedError 换成你的代码，再运行本格。


<details><summary>参考答案（先独立完成，再对照）</summary>

```python
import torch

def train_regressor(model, x, y, steps, lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for _ in range(steps):
        loss = torch.nn.functional.mse_loss(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return history
```

</details>


### 练习 4：软更新 target network

实现 `soft_update(target, source, tau)`：对每个参数执行 $\theta_t\leftarrow\tau\theta_s+(1-\tau)\theta_t$，原地修改 `target`，不能产生梯度。

<details><summary>提示（卡住 10 分钟再看）</summary>

`with torch.no_grad(): for t, s in zip(target.parameters(), source.parameters()): t.mul_(1 - tau).add_(tau * s)`。

</details>


In [11]:
import torch

def soft_update(target, source, tau):
    raise NotImplementedError


# ---- 自动检查（不用修改） ----
import torch

def test_soft_update():
    source, target = torch.nn.Linear(2, 1), torch.nn.Linear(2, 1)
    with torch.no_grad():
        source.weight.fill_(1.0); source.bias.fill_(1.0); target.weight.fill_(0.0); target.bias.fill_(0.0)
    soft_update(target, source, 0.1)
    assert torch.allclose(target.weight, torch.full_like(target.weight, 0.1)), "一次 tau=0.1 后应为 0.1"
    soft_update(target, source, 0.1)
    assert torch.allclose(target.bias, torch.tensor([0.19])), "两次后应为 0.1 + 0.1*(1-0.1) = 0.19"
    assert target.weight.grad is None, "软更新不应产生梯度"

check(test_soft_update)


[未完成] test_soft_update：把 raise NotImplementedError 换成你的代码，再运行本格。


<details><summary>参考答案（先独立完成，再对照）</summary>

```python
import torch

def soft_update(target, source, tau):
    with torch.no_grad():
        for t, s in zip(target.parameters(), source.parameters()):
            t.mul_(1 - tau).add_(tau * s)
```

</details>


## 初学者常见坑与过关标准

- 把 batch 维 `squeeze` 掉，单样本能跑、多样本失败。
- 忘记 `zero_grad()`；对 target 路径忘记 `detach/no_grad()`。
- NumPy 默认 float64，网络参数通常 float32，dtype 不一致。
- 保存整个临时计算图导致内存持续增长。

**练习：**把回归模型改成一层线性网络；比较 SGD 和 Adam，但保持数据、初始化、步数一致。

**过关：**能不看模板写出完整训练循环；能解释何时使用 `detach`；会检查 tensor shape、dtype、device 和 gradient。

**官方选读：**[PyTorch Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)，依次看 Tensors、Build Model、Autograd、Optimization、Save/Load。
